# Como baixar dados públicos da Binance e construir o dataset `btc_binance_ml_dataset.csv`

Este notebook mostra, passo a passo, como baixar os candles diários públicos do par **BTCUSDT** na Binance Public Data, processar os arquivos e gerar uma base final para um MVP de Machine Learning.

O objetivo final é criar um CSV semelhante a:

```text
btc_binance_ml_dataset.csv
```

Esse arquivo contém dados históricos do Bitcoin, variáveis derivadas e a variável-alvo `high_future_volatility`, usada para classificar se os 30 dias seguintes foram um período de alta volatilidade.

> Observação: este notebook não usa API key, credenciais privadas ou dados internos. Ele usa apenas arquivos públicos da Binance.

## 1. Instalação e importação das bibliotecas

As bibliotecas usadas são comuns em projetos de dados: `pandas` para manipulação de dados, `requests` para baixar arquivos, `zipfile` para ler os arquivos compactados e `pathlib` para lidar com caminhos.

In [ ]:
import zipfile
from pathlib import Path

import pandas as pd
import requests

## 2. Configuração do download

A Binance Public Data organiza os arquivos por mercado, par, intervalo e mês. Neste exemplo, serão usados candles diários (`1d`) do par `BTCUSDT` no mercado spot.

Fonte original dos dados:

https://data.binance.vision/?prefix=data/spot/monthly/klines/BTCUSDT/1d/

Documentação:

https://github.com/binance/binance-public-data

In [ ]:
SYMBOL = "BTCUSDT"
INTERVAL = "1d"
START_YEAR = 2018
END_YEAR = 2025

BASE_URL = "https://data.binance.vision/data/spot/monthly/klines"

RAW_DIR = Path("data/binance/raw")
PROCESSED_DIR = Path("data/binance/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DAILY_OUTPUT_PATH = PROCESSED_DIR / "btc_binance_daily.csv"
ML_OUTPUT_PATH = PROCESSED_DIR / "btc_binance_ml_dataset.csv"

## 3. Função para baixar os arquivos mensais

Cada arquivo mensal possui um nome no formato:

```text
BTCUSDT-1d-2024-01.zip
```

A função abaixo baixa o arquivo se ele ainda não existir localmente. Se o arquivo já existir, ele é ignorado para evitar downloads repetidos.

In [ ]:
def download_file(url: str, output_path: Path) -> bool:
	if output_path.exists():
		print("Skipping existing file:", output_path)
		return True

	print("Downloading:", url)

	response = requests.get(url, timeout=60)

	if response.status_code == 404:
		print("File not found:", url)
		return False

	response.raise_for_status()
	output_path.write_bytes(response.content)

	return True

## 4. Função para ler cada arquivo ZIP

Os arquivos da Binance vêm compactados em `.zip`. Dentro de cada `.zip`, há um arquivo `.csv` sem cabeçalho. Por isso, os nomes das colunas são definidos manualmente.

In [ ]:
def read_zip_csv(zip_path: Path) -> pd.DataFrame:
	with zipfile.ZipFile(zip_path) as zf:
		csv_names = [name for name in zf.namelist() if name.endswith(".csv")]

		if not csv_names:
			raise RuntimeError(f"No CSV found inside {zip_path}")

		with zf.open(csv_names[0]) as file:
			df = pd.read_csv(file, header=None)

	df.columns = [
		"open_time",
		"open",
		"high",
		"low",
		"close",
		"volume",
		"close_time",
		"quote_asset_volume",
		"number_of_trades",
		"taker_buy_base_asset_volume",
		"taker_buy_quote_asset_volume",
		"ignore",
	]

	return df

## 5. Tratamento dos timestamps

Arquivos mais antigos da Binance usam timestamps em **milissegundos**. Arquivos mais recentes de spot podem usar **microssegundos**. Se tentarmos ler microssegundos como milissegundos, o pandas pode interpretar datas absurdamente futuras.

A função abaixo converte os dois formatos para milissegundos.

In [ ]:
def normalize_timestamp_to_ms(series: pd.Series) -> pd.Series:
	series = pd.to_numeric(series, errors="coerce")

	# Timestamps em microssegundos são muito maiores que timestamps em milissegundos.
	# Quando o valor for maior que 10**14, dividimos por 1000 para converter para ms.
	return series.where(series < 10**14, series // 1000)

## 6. Baixar e combinar os candles diários

Esta etapa percorre todos os meses entre `START_YEAR` e `END_YEAR`, baixa os arquivos disponíveis, lê os CSVs e combina tudo em uma única base diária.

In [ ]:
frames = []

for year in range(START_YEAR, END_YEAR + 1):
	for month in range(1, 13):
		filename = f"{SYMBOL}-{INTERVAL}-{year}-{month:02d}.zip"
		url = f"{BASE_URL}/{SYMBOL}/{INTERVAL}/{filename}"
		zip_path = RAW_DIR / filename

		ok = download_file(url, zip_path)

		if not ok:
			continue

		df_month = read_zip_csv(zip_path)
		frames.append(df_month)

if not frames:
	raise RuntimeError("No data files were downloaded/read.")

raw_df = pd.concat(frames, ignore_index=True)

print("Rows loaded:", len(raw_df))
raw_df.head()

## 7. Limpeza e seleção das colunas principais

Aqui os timestamps são convertidos para datas, as colunas numéricas são convertidas para tipo numérico e a base final diária é organizada.

In [ ]:
df_daily = raw_df.copy()

# Normalização dos timestamps
df_daily["open_time_ms"] = normalize_timestamp_to_ms(df_daily["open_time"])
df_daily["close_time_ms"] = normalize_timestamp_to_ms(df_daily["close_time"])

# Criação da coluna date
df_daily["date"] = pd.to_datetime(
	df_daily["open_time_ms"],
	unit="ms",
	utc=True,
	errors="coerce"
).dt.strftime("%Y-%m-%d")

numeric_cols = [
	"open",
	"high",
	"low",
	"close",
	"volume",
	"quote_asset_volume",
	"number_of_trades",
	"taker_buy_base_asset_volume",
	"taker_buy_quote_asset_volume",
]

for col in numeric_cols:
	df_daily[col] = pd.to_numeric(df_daily[col], errors="coerce")

df_daily = df_daily[
	[
		"date",
		"open",
		"high",
		"low",
		"close",
		"volume",
		"quote_asset_volume",
		"number_of_trades",
		"taker_buy_base_asset_volume",
		"taker_buy_quote_asset_volume",
	]
]

df_daily = (
	df_daily
	.dropna()
	.drop_duplicates(subset=["date"])
	.sort_values("date")
	.reset_index(drop=True)
)

df_daily.to_csv(DAILY_OUTPUT_PATH, index=False)

print("Daily file saved:", DAILY_OUTPUT_PATH)
print("Rows:", len(df_daily))
print("Start date:", df_daily["date"].min())
print("End date:", df_daily["date"].max())

df_daily.head()

## 8. Verificação da base diária

Antes de criar variáveis de Machine Learning, é importante verificar se há datas faltando, valores ausentes ou duplicatas.

In [ ]:
df_check = df_daily.copy()
df_check["date"] = pd.to_datetime(df_check["date"])

expected_dates = pd.date_range(
	start=df_check["date"].min(),
	end=df_check["date"].max(),
	freq="D"
)

missing_dates = expected_dates.difference(df_check["date"])

print("Rows:", len(df_check))
print("Start date:", df_check["date"].min())
print("End date:", df_check["date"].max())
print("Expected daily rows:", len(expected_dates))
print("Missing dates:", len(missing_dates))
print("Missing values:", df_check.isna().sum().sum())
print("Duplicated dates:", df_check["date"].duplicated().sum())

## 9. Criação das variáveis para Machine Learning

Nesta etapa, são criadas variáveis derivadas a partir dos dados históricos:

- retornos passados
- volatilidade histórica
- médias móveis de volume
- variações de volume
- médias móveis do preço
- distância em relação às médias móveis
- amplitude diária

Essas variáveis representam informações disponíveis até cada data de referência.

In [ ]:
def build_ml_dataset(df_daily: pd.DataFrame, train_cutoff: str = "2024-01-01") -> tuple[pd.DataFrame, float]:
	df = df_daily.copy()

	df["date"] = pd.to_datetime(df["date"])
	df = df.sort_values("date").reset_index(drop=True)

	numeric_cols = [
		"open",
		"high",
		"low",
		"close",
		"volume",
		"quote_asset_volume",
		"number_of_trades",
		"taker_buy_base_asset_volume",
		"taker_buy_quote_asset_volume",
	]

	for col in numeric_cols:
		df[col] = pd.to_numeric(df[col], errors="coerce")

	df = df.dropna(subset=numeric_cols).reset_index(drop=True)

	# Retornos passados
	df["return_1d"] = df["close"].pct_change(1)
	df["return_7d"] = df["close"].pct_change(7)
	df["return_30d"] = df["close"].pct_change(30)
	df["return_90d"] = df["close"].pct_change(90)
	df["return_365d"] = df["close"].pct_change(365)

	# Volatilidade passada baseada no retorno diário
	df["volatility_7d"] = df["return_1d"].rolling(7).std()
	df["volatility_30d"] = df["return_1d"].rolling(30).std()
	df["volatility_90d"] = df["return_1d"].rolling(90).std()
	df["volatility_365d"] = df["return_1d"].rolling(365).std()

	# Volume negociado
	df["volume_mean_7d"] = df["volume"].rolling(7).mean()
	df["volume_mean_30d"] = df["volume"].rolling(30).mean()
	df["volume_mean_90d"] = df["volume"].rolling(90).mean()

	df["volume_change_7d"] = df["volume_mean_7d"] / df["volume_mean_30d"] - 1
	df["volume_change_30d"] = df["volume_mean_30d"] / df["volume_mean_90d"] - 1

	# Volume em USDT
	df["quote_volume_mean_7d"] = df["quote_asset_volume"].rolling(7).mean()
	df["quote_volume_mean_30d"] = df["quote_asset_volume"].rolling(30).mean()
	df["quote_volume_mean_90d"] = df["quote_asset_volume"].rolling(90).mean()

	df["quote_volume_change_7d"] = df["quote_volume_mean_7d"] / df["quote_volume_mean_30d"] - 1
	df["quote_volume_change_30d"] = df["quote_volume_mean_30d"] / df["quote_volume_mean_90d"] - 1

	# Número de trades
	df["trades_mean_7d"] = df["number_of_trades"].rolling(7).mean()
	df["trades_mean_30d"] = df["number_of_trades"].rolling(30).mean()
	df["trades_mean_90d"] = df["number_of_trades"].rolling(90).mean()

	df["trades_change_7d"] = df["trades_mean_7d"] / df["trades_mean_30d"] - 1
	df["trades_change_30d"] = df["trades_mean_30d"] / df["trades_mean_90d"] - 1

	# Amplitude diária
	df["hl_range"] = (df["high"] - df["low"]) / df["close"]
	df["hl_range_7d"] = df["hl_range"].rolling(7).mean()
	df["hl_range_30d"] = df["hl_range"].rolling(30).mean()
	df["hl_range_90d"] = df["hl_range"].rolling(90).mean()

	# Médias móveis
	df["ma_7d"] = df["close"].rolling(7).mean()
	df["ma_30d"] = df["close"].rolling(30).mean()
	df["ma_90d"] = df["close"].rolling(90).mean()
	df["ma_365d"] = df["close"].rolling(365).mean()

	df["distance_ma_7d"] = df["close"] / df["ma_7d"] - 1
	df["distance_ma_30d"] = df["close"] / df["ma_30d"] - 1
	df["distance_ma_90d"] = df["close"] / df["ma_90d"] - 1
	df["distance_ma_365d"] = df["close"] / df["ma_365d"] - 1

	# Volatilidade futura dos próximos 30 dias.
	# O shift(-1) faz o cálculo começar no dia seguinte.
	df["future_volatility_30d"] = (
		df["return_1d"]
		.shift(-1)
		.rolling(30)
		.std()
		.shift(-29)
	)

	# O limite de alta volatilidade é calculado apenas no período de treino.
	# Neste MVP, usamos o quantil 0.60 para representar volatilidade acima do padrão histórico.
	train_mask = df["date"] < train_cutoff
	threshold = df.loc[train_mask, "future_volatility_30d"].quantile(0.60)

	df["high_future_volatility"] = (
		df["future_volatility_30d"] >= threshold
	).astype(int)

	df = df.dropna().reset_index(drop=True)

	return df, threshold

## 10. Gerar e salvar o dataset final

Agora a função é aplicada à base diária e o resultado é salvo como `btc_binance_ml_dataset.csv`.

In [ ]:
df_ml, threshold = build_ml_dataset(df_daily, train_cutoff="2024-01-01")

df_ml.to_csv(ML_OUTPUT_PATH, index=False)

print("ML dataset saved:", ML_OUTPUT_PATH)
print("Rows:", len(df_ml))
print("Columns:", len(df_ml.columns))
print("Start date:", df_ml["date"].min())
print("End date:", df_ml["date"].max())
print("High future volatility threshold:", threshold)

print("\nTarget distribution:")
print(df_ml["high_future_volatility"].value_counts())
print(df_ml["high_future_volatility"].value_counts(normalize=True))

df_ml.head()

## 11. Conferência final

A base final deve ter uma estrutura próxima à usada no MVP principal, com colunas originais, variáveis derivadas e a variável-alvo `high_future_volatility`.

A coluna `future_volatility_30d` foi mantida para auditoria da construção do alvo, mas **não deve ser usada como feature** no modelo, pois contém informação futura.

In [ ]:
print("Colunas do dataset final:")
print(df_ml.columns.tolist())

print("\nValores ausentes:")
print(df_ml.isna().sum().sum())

print("\nDuplicatas:")
print(df_ml.duplicated().sum())

## 12. Como usar o CSV em outro notebook

Depois de gerar o arquivo, ele pode ser carregado diretamente em outro notebook:

In [ ]:
df = pd.read_csv("data/binance/processed/btc_binance_ml_dataset.csv")
df["date"] = pd.to_datetime(df["date"])

df.head()

## 13. Observações metodológicas

1. A base final começa em 2019 porque algumas variáveis usam janelas históricas de até 365 dias.
2. A base termina antes do último dia disponível porque a variável-alvo depende dos 30 dias seguintes.
3. O limiar da variável-alvo é calculado usando apenas o período de treino, para reduzir risco de vazamento de dados.
4. A coluna `future_volatility_30d` não deve entrar no modelo.
5. O dataset usa apenas dados públicos de mercado e não contém dados pessoais ou informações privadas.